# LINDAS Triple Store Operations

This notebook contains code to upload RDF data to the LINDAS triple store and also to delete it.

## Load Environment Variables

In [2]:
from dotenv import load_dotenv
import os

ENV = "INT"

# Load the env file
load_dotenv(f"../.env.{ENV.lower()}")

print(os.getenv('LINDAS_USERNAME'))
print(os.getenv('GRAPH'))

lindas-fch-political-party
https://lindas.admin.ch/fch/political-party


## Upload Graph

In [3]:
import requests
import os
from pathlib import Path
from requests.auth import HTTPBasicAuth

# Read ENV variables
USERNAME = os.getenv('LINDAS_USERNAME')
PASSWORD = os.getenv('LINDAS_PASSWORD')
GRAPH = os.getenv("GRAPH")
ENDPOINT = os.getenv("ENDPOINT")

params = {
    'graph': GRAPH
}

# Upload the new data

# Define the headers
headers = {
    'Content-Type': 'text/turtle',
}

folder = Path("../data/output")
files = ttl_files = [f.name for f in folder.glob("*.ttl")]

for i, file in enumerate(files):
    if not os.path.exists(f"../data/output/{file}"):
        print(f"File {file} does not exist.")
        continue
    print(f"Uploading {file} to graph {GRAPH}")

    with open(f"../data/output/{file}", 'r') as f:
        # Make the PUT request with the file data, headers, and authentication
        
        if i == 0:
            # Use PUT for the first file to delete existing data
            response = requests.put(ENDPOINT, params=params, headers=headers, data=f, auth=HTTPBasicAuth(USERNAME, PASSWORD))
        else:
            # Use POST for subsequent files to append data
            response = requests.post(ENDPOINT, params=params, headers=headers, data=f, auth=HTTPBasicAuth(USERNAME, PASSWORD))

    if response.status_code == 204:
        print(f"Upload succeeded for {file} with status code {response.status_code}")
    else:
        print(f"Upload failed for {file} with status code {response.status_code}")
        print(response.text)

Uploading parties.ttl to graph https://lindas.admin.ch/fch/political-party
Upload succeeded for parties.ttl with status code 204


## Delete Graph

In [4]:
import requests
import os
from requests.auth import HTTPBasicAuth

# Read ENV variables
USERNAME = os.getenv('LINDAS_USERNAME')
PASSWORD = os.getenv('LINDAS_PASSWORD')
GRAPH = os.getenv("GRAPH")
ENDPOINT = os.getenv("ENDPOINT")

params = {
    'graph': GRAPH
}

# Delete the existing graph if it exists
delete_response = requests.delete(ENDPOINT, params=params, auth=HTTPBasicAuth(USERNAME, PASSWORD), timeout=60)

if delete_response.status_code == 204:
    print(f'Graph {GRAPH} deleted successfully.')
else: 
    print(f"Failed to delete {GRAPH} with status code {delete_response.status_code}")

Graph https://lindas.admin.ch/fch/political-party deleted successfully.
